# Video JEPA - Level 0: The Highest Overview

**What changed from Image JEPA?**

Image JEPA had **no predictor** — it was a Joint Embedding Architecture (two views → same representation).

Video JEPA adds a **predictor**: given past frames, predict future frame representations in latent space.

The pipeline:
1. Load data (Moving MNIST — bouncing digits)
2. Build model (Encoder + **Predictor** + JEPA wrapper)
3. Train (prediction loss + VC regularization)
4. Evaluate (reconstruct pixels + detect digit locations)

In [ ]:
import sys
from pathlib import Path

ROOT = str(Path.cwd().parents[1]) if Path.cwd().name == "my_video_jepa" else str(Path.cwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from eb_jepa.architectures import ResNet5, ResUNet, StateOnlyPredictor, Projector, DetHead
from eb_jepa.datasets.moving_mnist import MovingMNISTDet
from eb_jepa.image_decoder import ImageDecoder
from eb_jepa.jepa import JEPA, JEPAProbe
from eb_jepa.losses import SquareLossSeq, VCLoss

## Step 1 — Data

**Moving MNIST**: 10,000 sequences of two digits bouncing inside a 64×64 box.

No augmentation needed — the temporal dynamics *are* the learning signal.

Each sample returns:
- `video`: `[1, T, 64, 64]` — grayscale frames
- `digit_location`: `[T, 8, 8]` — coarse heatmap of where the digits are (for eval only)

In [ ]:
train_set = MovingMNISTDet(split="train")
val_set   = MovingMNISTDet(split="val")

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} sequences | Val: {len(val_set)} sequences")

sample = train_set[0]
print(f"Video shape:          {sample['video'].shape}")        # [1, T, 64, 64]
print(f"Digit location shape: {sample['digit_location'].shape}")  # [T, 8, 8]

In [ ]:
import matplotlib.pyplot as plt

video = sample["video"]  # [1, T, 64, 64]
T = video.shape[1]

fig, axes = plt.subplots(1, min(T, 5), figsize=(12, 3))
for t, ax in enumerate(axes):
    ax.imshow(video[0, t].numpy(), cmap="gray")
    ax.set_title(f"t={t}")
    ax.axis("off")
fig.suptitle("Moving MNIST — first 5 frames", fontsize=14)
plt.tight_layout()
plt.show()

## Step 2 — Model

This is where Video JEPA diverges from Image JEPA. Three new pieces:

| Component | What it does | Architecture |
|-----------|-------------|-------------|
| **Encoder** | Encodes each frame → spatial feature map `[B, 16, T, H, W]` | ResNet5 (5-layer lightweight ResNet) |
| **Predictor** | Takes `[prev_frame, next_frame]` representations → predicts next | ResUNet (UNet with skip connections) |
| **JEPA wrapper** | Orchestrates multi-step unrolling + loss computation | `JEPA(encoder, predictor, regularizer, pred_loss)` |

The predictor is wrapped in `StateOnlyPredictor` — it ignores actions (no actions in this example) and concatenates consecutive frames channel-wise before feeding to the ResUNet.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Dimensions (from default.yaml) ---
D_OBS = 1    # input channels (grayscale)
H_ENC = 32   # hidden dim in encoder
D_STC = 16   # representation dim (encoder output channels)
H_PRE = 32   # hidden dim in predictor

# --- Encoder: each frame → spatial feature map ---
encoder = ResNet5(D_OBS, H_ENC, D_STC)

# --- Predictor: [prev, next] representations → predicted next ---
predictor_net = ResUNet(2 * D_STC, H_PRE, D_STC)        # input is 2x D_STC (two frames concatenated)
predictor     = StateOnlyPredictor(predictor_net, context_length=2)

# --- Projector: used inside the losses (not on the encoder output) ---
projector = Projector(f"{D_STC}-{D_STC*4}-{D_STC*4}")   # "16-64-64"

# --- Losses ---
regularizer = VCLoss(std_coeff=10.0, cov_coeff=100.0, proj=projector)  # prevents collapse
pred_loss   = SquareLossSeq(projector)                                  # MSE in projected space

# --- JEPA: ties everything together ---
jepa = JEPA(encoder, encoder, predictor, regularizer, pred_loss).to(device)

print(f"Encoder params:   {sum(p.numel() for p in encoder.parameters()):,}")
print(f"Predictor params: {sum(p.numel() for p in predictor.parameters()):,}")
print(f"Total params:     {sum(p.numel() for p in jepa.parameters()):,}")
print(f"Device: {device}")

### Eval probes (trained alongside, but encoder stays frozen for these)

Two probes decode the learned representations back to pixel space — purely for monitoring:

- **Pixel decoder**: can the representation reconstruct the original frames?
- **Detection head**: can the representation locate the digits?

These don't affect the JEPA training — the encoder gradients are detached.

In [ ]:
decoder = ImageDecoder(D_STC, D_OBS)
dethead = DetHead(D_STC, H_PRE, D_OBS)

pixel_decoder  = JEPAProbe(jepa, decoder, nn.MSELoss()).to(device)
detection_head = JEPAProbe(jepa, dethead, nn.BCELoss()).to(device)

## Step 3 — Optimizer

Adam with a lower learning rate for the pixel decoder (to prevent it from overfitting).

Compare with Image JEPA which used LARS + cosine warmup — video JEPA keeps it simple.

In [ ]:
from torch.optim import Adam

LR = 1e-3
NUM_EPOCHS = 5  # just a taste — real training uses 50
PRED_STEPS = 4  # how many steps ahead to predict during training

optimizer = Adam([
    {"params": jepa.parameters(),              "lr": LR},
    {"params": pixel_decoder.head.parameters(), "lr": LR / 10},
    {"params": detection_head.head.parameters(),"lr": LR},
])

## Step 4 — Train

The key call is `jepa.unroll()` — this is where the magic happens:

1. **Encode** all frames → `[B, 16, T, H, W]` spatial features
2. **Predict** future frames in latent space (4 steps ahead)
3. **Compute losses**:
   - **Prediction loss**: MSE between predicted and actual representations (in projected space)
   - **VC regularization**: variance + covariance on the representations (prevents collapse)

Plus the two eval probe losses (reconstruction + detection) which train on frozen features.

In [ ]:
from tqdm.auto import tqdm

jepa.train()
pixel_decoder.train()
detection_head.train()

for epoch in range(NUM_EPOCHS):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for batch in pbar:
        x       = batch["video"].to(device)            # [B, 1, T, 64, 64]
        loc_map = batch["digit_location"].to(device)    # [B, T, 8, 8]

        optimizer.zero_grad()

        # --- JEPA forward: encode → predict → compute losses ---
        _, (jepa_loss, vc_loss, _, vc_dict, pred_loss) = jepa.unroll(
            x,
            actions=None,
            nsteps=PRED_STEPS,
            unroll_mode="parallel",
            compute_loss=True,
            return_all_steps=False,
        )

        # --- Eval probe losses (encoder is frozen for these) ---
        recon_loss = pixel_decoder(x, x)
        det_loss   = detection_head(x, loc_map)

        total_loss = jepa_loss + recon_loss + det_loss
        total_loss.backward()
        optimizer.step()

        pbar.set_postfix(jepa=f"{jepa_loss:.4f}", vc=f"{vc_loss:.4f}", pred=f"{pred_loss:.4f}")

    print(f"Epoch {epoch} done | jepa_loss={jepa_loss:.4f}  vc={vc_loss:.4f}  pred={pred_loss:.4f}")

## Step 5 — Evaluate

Freeze the model, unroll predictions over the full sequence, and visualize:
- **Row 1**: Ground truth frames
- **Row 2**: Reconstructed frames from predicted representations
- **Row 3**: Digit location predictions overlaid on reconstructions

In [ ]:
import torch.nn.functional as F

jepa.eval()
pixel_decoder.eval()
detection_head.eval()

# Grab one batch
val_batch = next(iter(val_loader))
x = val_batch["video"].to(device)  # [B, 1, T, 64, 64]

with torch.no_grad():
    # Encode all frames
    x_enc = jepa.encoder(x)  # [B, 16, T, H, W]

    T = x.shape[2]
    # Unroll predictions for T-2 steps (keeping first 2 as context)
    preds, _ = jepa.unroll(
        x, actions=None, nsteps=T - 2,
        unroll_mode="parallel", compute_loss=False, return_all_steps=True,
    )

    # Build multi-step rollout: use ground truth for context, predictions for the rest
    rollout = x_enc[:, :, 1:].clone()
    for t in range(1, T - 1):
        rollout[:, :, t:] = preds[t - 1][:, :, t - 1:]

    # Decode rollout back to pixels
    rollout_recon = pixel_decoder.head(rollout)

    # Detection predictions
    loc_pred = detection_head.head(rollout)
    loc_pred = F.interpolate(loc_pred, (x.shape[-2], x.shape[-1]), mode="nearest")

# --- Visualize one sample ---
idx = 0
gt_frames      = x[idx, 0, 1:].cpu().numpy()             # [T-1, 64, 64]
recon_frames   = rollout_recon[idx, 0].clamp(0, 1).cpu().numpy()  # [T-1, 64, 64]
det_frames     = loc_pred[idx, 0].cpu().numpy()            # [T-1, 64, 64]

n_show = min(4, gt_frames.shape[0])
fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
for t in range(n_show):
    axes[0, t].imshow(gt_frames[t], cmap="gray"); axes[0, t].set_title(f"t={t+1}")
    axes[1, t].imshow(recon_frames[t], cmap="gray")
    axes[2, t].imshow(det_frames[t], cmap="Blues")
    for row in range(3):
        axes[row, t].axis("off")

axes[0, 0].set_ylabel("Ground truth",  fontsize=12)
axes[1, 0].set_ylabel("Predicted",     fontsize=12)
axes[2, 0].set_ylabel("Detection",     fontsize=12)
fig.suptitle("Video JEPA — rollout predictions vs ground truth", fontsize=14)
plt.tight_layout()
plt.show()

---

That's the whole thing. Compared to Image JEPA:

| | Image JEPA | Video JEPA |
|---|---|---|
| **Data** | Static images, 2 augmented views | Video sequences, no augmentation |
| **Predictor** | None | ResUNet (spatial, multi-step) |
| **Loss** | VICReg (invariance + var + cov) | Prediction MSE + VC regularization |
| **Eval** | Linear probe (classification) | Pixel reconstruction + digit detection |
| **Key idea** | Two views of same image should map close | Future frames should be predictable from past |

Next levels will zoom into each piece — especially `jepa.unroll()` and how multi-step prediction works.